[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/ab-test-practice/blob/main/study_notes/week04_study_concurrent_tests.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 4 Study Guide: Running Concurrent A/B Tests

**Course:** A/B Testing in Practice ()  
**Company Context:** FamilyNest — Airbnb for families with young kids  
**Role:** Product Data Scientist  

---

## Motivation

At FamilyNest, we have multiple product teams working simultaneously:
- The **Search & Discovery** team wants to test a new ranking algorithm for family-friendly listings
- The **Trust & Safety** team wants to test adding a "verified child-safe" badge on the same listing cards
- The **Booking Flow** team has two ideas for the checkout page: a price breakdown redesign AND a new cancellation policy display

How do we run all of these experiments without waiting months? That's the challenge of concurrent testing.

---

## 1. Running Multiple Tests Simultaneously

### Why this comes up

Product teams want to move fast. At FamilyNest, we might have 5-10 experiment ideas in the pipeline at any given time. If each test takes 2-4 weeks, running them sequentially means months of delay.

### The core challenge

If two tests modify the **same surface** (e.g., the same page or the same UI element), they may **interact** — the effect of one change could depend on whether the other change is also present.

### Key distinction

| Scenario | Risk of Interaction | Action |
|----------|-------------------|--------|
| Tests on **different surfaces** (e.g., search page vs. checkout page) | Very low | Safe to run concurrently without special handling |
| Tests on the **same surface** (e.g., two changes to the listing card) | High | Need careful handling — see options below |

**FamilyNest Example:**
- Testing a new homepage hero image AND a new checkout flow → Different surfaces → run simultaneously, no worries
- Testing a new listing card layout AND a new badge on listing cards → Same surface → need to think carefully

---

## 2. Test Structure Options for Concurrent Changes

When you have **TWO changes for the SAME page/surface**, you have four main options.

### FamilyNest Scenario

We want to test two changes on the **listing detail page**:
- **Change A:** New photo carousel with kid-focused photos prioritized
- **Change B:** New "Family Amenities" section placed higher on the page

### Option A: Sequential Testing

Run Test 1 first, get results, then run Test 2.

```
Week 1-3: Test A (photo carousel)
    → Analyze results
    → Ship if positive
Week 4-6: Test B (family amenities section)
    → Analyze results
    → Ship if positive
```

| Pros | Cons |
|------|------|
| Clean, no interaction concerns | Slow — takes 2x the time |
| Simple analysis | Second test's baseline may change if first test ships |
| Easy to explain to stakeholders | Opportunity cost of delay |

### Option B: Mutual Exclusion (Traffic Splitting)

Split traffic so no user sees both tests. Each test has its own control/treatment split within its slice.

```
Total traffic: 100%
├── Test A slice: 50% of traffic
│   ├── Control A: 25% of total
│   └── Treatment A: 25% of total
└── Test B slice: 50% of traffic
    ├── Control B: 25% of total
    └── Treatment B: 25% of total
```

| Pros | Cons |
|------|------|
| Run simultaneously, no interactions possible | Each test has half the traffic |
| Clean causal inference | Need longer to reach power OR can only detect larger effects |
| Tests are independent | More complex traffic allocation system needed |

### Option C: Full Factorial (Multi-arm)

Test ALL combinations. Every user is assigned to one of four groups:

```
Total traffic: 100%
├── Control (neither A nor B): 25%
├── Treatment A only (photo carousel): 25%
├── Treatment B only (amenities section): 25%
└── Treatment A+B (both changes): 25%
```

| Pros | Cons |
|------|------|
| Can detect interactions between A and B | Most complex to set up and analyze |
| Fastest path to learning if you have traffic | Needs more traffic per cell |
| Answers "should we ship both together?" | 4 groups means each gets only 25% of traffic |

### Option D: Overlapping Tests (with Interaction Checks)

Both tests run on 100% of traffic independently. Users can be in both tests.

```
Test A: 50% control / 50% treatment (all users)
Test B: 50% control / 50% treatment (all users)

Result: users end up in one of four combinations by chance:
├── Control A + Control B: ~25%
├── Treatment A + Control B: ~25%
├── Control A + Treatment B: ~25%
└── Treatment A + Treatment B: ~25%
```

| Pros | Cons |
|------|------|
| Full power for each test | If interactions exist, results are biased |
| No traffic splitting needed | Must CHECK for interactions after |
| Simple to set up | If interaction found, harder to interpret results |

---

## 3. Sample Size for Multi-arm Tests

When you have more than two groups (variants), the sample size requirements change.

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.power import NormalIndPower

def calculate_sample_size_per_variant(
    baseline_rate,
    mde_relative,
    alpha=0.05,
    power=0.8,
    num_comparisons=1
):
    """
    Calculate sample size per variant, adjusting for multiple comparisons.
    
    Parameters
    ----------
    baseline_rate : float
        Baseline conversion rate (e.g., 0.05 for 5%)
    mde_relative : float
        Minimum detectable effect as relative change (e.g., 0.10 for 10% relative lift)
    alpha : float
        Significance level (before correction)
    power : float
        Statistical power
    num_comparisons : int
        Number of pairwise comparisons (for Bonferroni correction)
    
    Returns
    -------
    int
        Required sample size per variant
    """
    # Apply Bonferroni correction
    adjusted_alpha = alpha / num_comparisons
    
    # Calculate absolute effect size
    treatment_rate = baseline_rate * (1 + mde_relative)
    
    # Pooled standard error for proportions
    p_pooled = (baseline_rate + treatment_rate) / 2
    effect_size = abs(treatment_rate - baseline_rate) / np.sqrt(p_pooled * (1 - p_pooled))
    
    # Use statsmodels power analysis
    analysis = NormalIndPower()
    n = analysis.solve_power(
        effect_size=effect_size,
        alpha=adjusted_alpha,
        power=power,
        alternative='two-sided'
    )
    
    return int(np.ceil(n))

# FamilyNest example: listing detail page booking rate
baseline_booking_rate = 0.03  # 3% of listing page views result in a booking
mde = 0.10  # We want to detect a 10% relative lift (3% → 3.3%)

print("=== Sample Size Requirements for FamilyNest ===")
print(f"Baseline booking rate: {baseline_booking_rate:.1%}")
print(f"Minimum detectable effect: {mde:.0%} relative lift")
print(f"Target booking rate: {baseline_booking_rate * (1 + mde):.1%}")
print()

# Standard 2-arm test (1 comparison)
n_2arm = calculate_sample_size_per_variant(baseline_booking_rate, mde, num_comparisons=1)
print(f"2-arm test (control vs treatment):")
print(f"  Sample per variant: {n_2arm:,}")
print(f"  Total sample needed: {n_2arm * 2:,}")
print()

# 3-arm test (2 comparisons: A vs control, B vs control)
n_3arm = calculate_sample_size_per_variant(baseline_booking_rate, mde, num_comparisons=2)
print(f"3-arm test (control vs A, control vs B):")
print(f"  Sample per variant: {n_3arm:,}")
print(f"  Total sample needed: {n_3arm * 3:,}")
print()

# 4-arm full factorial (multiple comparisons)
n_4arm = calculate_sample_size_per_variant(baseline_booking_rate, mde, num_comparisons=3)
print(f"4-arm full factorial (3 pairwise comparisons vs control):")
print(f"  Sample per variant: {n_4arm:,}")
print(f"  Total sample needed: {n_4arm * 4:,}")

In [ ]:
# Duration calculation for FamilyNest

daily_listing_page_views = 15000  # daily unique visitors to listing detail pages

print("=== Test Duration Estimates ===")
print(f"Daily listing page traffic: {daily_listing_page_views:,} unique visitors")
print()

# 2-arm test
total_needed_2arm = n_2arm * 2
duration_2arm = np.ceil(total_needed_2arm / daily_listing_page_views)
print(f"2-arm test: {total_needed_2arm:,} total → {int(duration_2arm)} days")

# 3-arm test
total_needed_3arm = n_3arm * 3
duration_3arm = np.ceil(total_needed_3arm / daily_listing_page_views)
print(f"3-arm test: {total_needed_3arm:,} total → {int(duration_3arm)} days")

# 4-arm full factorial
total_needed_4arm = n_4arm * 4
duration_4arm = np.ceil(total_needed_4arm / daily_listing_page_views)
print(f"4-arm test: {total_needed_4arm:,} total → {int(duration_4arm)} days")

# Sequential approach (two 2-arm tests back-to-back)
duration_sequential = duration_2arm * 2
print(f"Sequential (two 2-arm tests): {int(duration_sequential)} days")

# Mutual exclusion (each test gets half traffic)
duration_mutual_excl = np.ceil(total_needed_2arm / (daily_listing_page_views / 2))
print(f"Mutual exclusion (each test gets half traffic): {int(duration_mutual_excl)} days")

print()
print("Key insight: Full factorial needs more total samples but")
print("gives you interaction information. Sequential is slowest.")

### Key Takeaways on Sample Size

- With **K variants**, you need more samples per variant due to multiple comparison corrections
- **Bonferroni correction**: Use `alpha / K` where K is the number of pairwise comparisons
- Duration formula:
  - 2-arm: `duration = (sample_per_variant * 2) / daily_traffic`
  - 3-arm: `duration = (sample_per_variant * 3) / daily_traffic`
  - K-arm: `duration = (sample_per_variant * K) / daily_traffic`
- Alternative: keep the same duration but accept you can only detect a larger MDE

---

## 4. Interaction Effects

### Definition

An **interaction effect** means the effect of Test A **depends on** whether Test B is also present.

### FamilyNest Example

- **Change A:** A new dynamic pricing algorithm that shows lower prices for longer stays
- **Change B:** A prominent "family discount" checkbox that applies 15% off

These might interact because:
- The pricing algorithm already lowers prices for the same segment the discount targets
- Together, prices might drop too low → hosts get unhappy → more cancellations
- OR together they might over-deliver on value perception → even higher conversion

The point is: the **combined effect** is not simply the sum of individual effects.

In [ ]:
import pandas as pd
np.random.seed(42)

# Simulate a full factorial experiment at FamilyNest
# Change A: Kid-focused photo carousel
# Change B: Family amenities section moved higher

n_per_group = 5000
baseline_rate = 0.03  # 3% booking rate

# Simulate WITH an interaction effect
# A alone lifts by 10% relative, B alone lifts by 8% relative
# But together they lift by 25% relative (more than additive = positive interaction)

groups = {
    'control': baseline_rate,                      # 3.0%
    'treatment_a': baseline_rate * 1.10,           # 3.3% (photo carousel)
    'treatment_b': baseline_rate * 1.08,           # 3.24% (amenities section)
    'treatment_ab': baseline_rate * 1.25           # 3.75% (both — more than additive!)
}

# Generate synthetic data
records = []
for variant, rate in groups.items():
    bookings = np.random.binomial(1, rate, n_per_group)
    for booking in bookings:
        records.append({'variant': variant, 'booked': booking})

df = pd.DataFrame(records)
print(f"Total observations: {len(df):,}")
print(f"\nBooking rates by variant:")
print(df.groupby('variant')['booked'].agg(['mean', 'count']).round(4))

In [ ]:
# How to test for interactions in a full factorial

metric = 'booked'

# Subset by variant
df_c = df[df['variant'] == 'control']
df_a = df[df['variant'] == 'treatment_a']
df_b = df[df['variant'] == 'treatment_b']
df_ab = df[df['variant'] == 'treatment_ab']

# Effect of A alone (compared to control)
effect_a = df_a[metric].mean() - df_c[metric].mean()

# Effect of A given B is present
effect_a_given_b = df_ab[metric].mean() - df_b[metric].mean()

# If these differ significantly → interaction exists
print("=== Interaction Check ===")
print(f"Effect of A alone:    {effect_a:.4f} ({effect_a/baseline_rate:.1%} relative)")
print(f"Effect of A given B:  {effect_a_given_b:.4f} ({effect_a_given_b/baseline_rate:.1%} relative)")
print(f"Difference (interaction): {effect_a_given_b - effect_a:.4f}")
print()

# The interaction effect: does A's effect change when B is present?
# Interaction = (AB - B) - (A - Control) = AB - B - A + Control
interaction = (df_ab[metric].mean() - df_b[metric].mean()) - (df_a[metric].mean() - df_c[metric].mean())
print(f"Interaction term: {interaction:.4f}")
print()

if abs(effect_a_given_b - effect_a) > 0.002:
    print("→ Possible interaction detected!")
    print("  The effect of the photo carousel DEPENDS on whether")
    print("  the family amenities section is also present.")
else:
    print("→ No meaningful interaction. Effects appear additive.")

In [ ]:
# Statistical test for interaction using regression
import statsmodels.api as sm

# Create indicator variables
df_factorial = df.copy()
df_factorial['has_a'] = df_factorial['variant'].isin(['treatment_a', 'treatment_ab']).astype(int)
df_factorial['has_b'] = df_factorial['variant'].isin(['treatment_b', 'treatment_ab']).astype(int)
df_factorial['interaction'] = df_factorial['has_a'] * df_factorial['has_b']

# Fit regression: booked ~ has_a + has_b + has_a*has_b
X = df_factorial[['has_a', 'has_b', 'interaction']]
X = sm.add_constant(X)
y = df_factorial['booked']

model = sm.OLS(y, X).fit()

print("=== Regression Results (Interaction Model) ===")
print(f"{'Term':<15} {'Coefficient':>12} {'p-value':>10} {'Significant?':>14}")
print("-" * 55)
for term, coef, pval in zip(model.params.index, model.params, model.pvalues):
    sig = "Yes *" if pval < 0.05 else "No"
    print(f"{term:<15} {coef:>12.4f} {pval:>10.4f} {sig:>14}")

print()
print("Interpretation:")
print(f"  - Intercept (baseline rate): {model.params['const']:.4f}")
print(f"  - Effect of A alone: {model.params['has_a']:.4f}")
print(f"  - Effect of B alone: {model.params['has_b']:.4f}")
print(f"  - Interaction (additional effect of having both): {model.params['interaction']:.4f}")
print()
if model.pvalues['interaction'] < 0.05:
    print("  → The interaction term IS statistically significant.")
    print("    The combined effect of A+B is NOT simply the sum of individual effects.")
else:
    print("  → The interaction term is NOT statistically significant.")
    print("    We cannot reject that effects are additive (may lack power to detect it).")

### How to Think About Interactions

**No interaction (additive effects):**
- Effect of A alone = +0.3pp
- Effect of B alone = +0.24pp
- Effect of A+B = +0.54pp (sum of parts)

**Positive interaction (synergy):**
- Effect of A alone = +0.3pp
- Effect of B alone = +0.24pp
- Effect of A+B = +0.75pp (more than sum → they amplify each other)

**Negative interaction (cannibalization):**
- Effect of A alone = +0.3pp
- Effect of B alone = +0.24pp
- Effect of A+B = +0.35pp (less than sum → they partially cancel each other)

---

## 5. Analyzing Multi-arm Experiments

### Key Principles

1. **Always compare each treatment to control** (not to each other, unless pre-specified)
2. **Apply multiple comparison correction** when making many comparisons
3. **Report each comparison separately** with its own p-value and confidence interval
4. **Pre-register** which comparisons you plan to make

In [ ]:
from scipy.stats import norm

def calculate_results_multiarm(df, metric, control_name='control', alpha=0.05):
    """
    Calculate A/B test results for a multi-arm experiment.
    Compares each non-control variant to control with Bonferroni correction.
    
    Parameters
    ----------
    df : DataFrame
        Must have columns: 'variant' and the metric column
    metric : str
        Name of the metric column (binary 0/1)
    control_name : str
        Name of the control variant
    alpha : float
        Overall significance level (will be Bonferroni-corrected)
    
    Returns
    -------
    DataFrame with results for each treatment vs control comparison
    """
    # Get all treatment variants
    treatments = [v for v in df['variant'].unique() if v != control_name]
    num_comparisons = len(treatments)
    adjusted_alpha = alpha / num_comparisons  # Bonferroni correction
    
    # Control stats
    control_data = df[df['variant'] == control_name][metric]
    n_control = len(control_data)
    p_control = control_data.mean()
    
    results = []
    
    for treatment in sorted(treatments):
        treatment_data = df[df['variant'] == treatment][metric]
        n_treatment = len(treatment_data)
        p_treatment = treatment_data.mean()
        
        # Absolute difference
        diff = p_treatment - p_control
        
        # Relative lift
        relative_lift = diff / p_control if p_control > 0 else 0
        
        # Standard error of the difference
        se = np.sqrt(p_control * (1 - p_control) / n_control + 
                     p_treatment * (1 - p_treatment) / n_treatment)
        
        # Z-statistic and p-value
        z_stat = diff / se
        p_value = 2 * (1 - norm.cdf(abs(z_stat)))
        
        # Confidence interval (using adjusted alpha for Bonferroni)
        z_crit = norm.ppf(1 - adjusted_alpha / 2)
        ci_lower = diff - z_crit * se
        ci_upper = diff + z_crit * se
        
        # Significance check (against adjusted alpha)
        is_significant = p_value < adjusted_alpha
        
        results.append({
            'comparison': f"{treatment} vs {control_name}",
            'n_control': n_control,
            'n_treatment': n_treatment,
            'rate_control': p_control,
            'rate_treatment': p_treatment,
            'absolute_diff': diff,
            'relative_lift': relative_lift,
            'se': se,
            'z_stat': z_stat,
            'p_value': p_value,
            f'ci_{int((1-adjusted_alpha)*100)}%_lower': ci_lower,
            f'ci_{int((1-adjusted_alpha)*100)}%_upper': ci_upper,
            'significant': is_significant
        })
    
    results_df = pd.DataFrame(results)
    
    # Print summary
    print(f"=== Multi-arm Test Results ===")
    print(f"Number of comparisons: {num_comparisons}")
    print(f"Bonferroni-adjusted alpha: {adjusted_alpha:.4f} (original: {alpha})")
    print(f"Control rate: {p_control:.4f} (n={n_control:,})")
    print()
    
    for _, row in results_df.iterrows():
        sig_marker = " ***" if row['significant'] else ""
        print(f"  {row['comparison']}:")
        print(f"    Rate: {row['rate_treatment']:.4f} (n={row['n_treatment']:,})")
        print(f"    Lift: {row['relative_lift']:+.2%} (absolute: {row['absolute_diff']:+.4f})")
        print(f"    p-value: {row['p_value']:.4f}{sig_marker}")
        print()
    
    return results_df

# Run the analysis on our factorial experiment
results = calculate_results_multiarm(df, 'booked', control_name='control', alpha=0.05)

In [ ]:
# Visualize the results
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

# Plot booking rates by variant
variant_order = ['control', 'treatment_a', 'treatment_b', 'treatment_ab']
variant_labels = ['Control', 'Photo Carousel\n(A only)', 'Amenities Section\n(B only)', 'Both A+B']

rates = [df[df['variant'] == v]['booked'].mean() for v in variant_order]
colors = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c']

bars = ax.bar(variant_labels, rates, color=colors, edgecolor='black', linewidth=0.5)

# Add rate labels on bars
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
            f'{rate:.2%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add additive prediction line for A+B
additive_prediction = rates[0] + (rates[1] - rates[0]) + (rates[2] - rates[0])
ax.axhline(y=additive_prediction, color='orange', linestyle='--', alpha=0.7,
           label=f'Additive prediction for A+B: {additive_prediction:.2%}')

ax.set_ylabel('Booking Rate', fontsize=12)
ax.set_title('FamilyNest Full Factorial: Listing Page Experiment', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, max(rates) * 1.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\nAdditive prediction for A+B: {additive_prediction:.4f}")
print(f"Actual A+B rate: {rates[3]:.4f}")
print(f"Difference (interaction): {rates[3] - additive_prediction:.4f}")
if rates[3] > additive_prediction:
    print("→ Positive interaction (synergy): combined effect exceeds sum of parts")
else:
    print("→ Negative interaction: combined effect is less than sum of parts")

---

## 6. Practical Decision Framework

Use this table when deciding how to handle concurrent tests at FamilyNest:

| Situation | Recommendation | Example |
|-----------|---------------|---------|
| Two changes on **different pages** | Run simultaneously (overlap), no concern | Search ranking + checkout flow |
| Two changes on **same page**, have traffic | **Full factorial** | Photo carousel + amenities section on listing page |
| Two changes on **same page**, low traffic | **Sequential** OR **mutual exclusion** | Two changes to the host dashboard (low-traffic page) |
| Interaction **detected** | Don't ship independently; ship the combination or pick one | Pricing algorithm + discount badge |
| No interaction **detected** | Can ship either/both independently | Changes that proved additive |

### Decision Flowchart (Mental Model)

```
Do tests touch the same surface?
├── NO → Run them overlapping. Done.
└── YES → Do you have enough traffic for a full factorial?
    ├── YES → Full factorial (best learning)
    └── NO → Is speed more important than detecting interactions?
        ├── YES → Overlapping + post-hoc interaction check
        └── NO → Sequential or mutual exclusion
```

In [ ]:
def recommend_concurrent_strategy(
    same_surface: bool,
    daily_traffic: int,
    baseline_rate: float,
    mde: float,
    max_days: int = 28
):
    """
    Recommend a strategy for running concurrent tests.
    
    Parameters
    ----------
    same_surface : bool
        Whether both tests affect the same page/surface
    daily_traffic : int
        Daily unique visitors to the surface
    baseline_rate : float
        Current conversion rate
    mde : float
        Minimum detectable effect (relative)
    max_days : int
        Maximum acceptable test duration
    """
    print("=== Concurrent Test Strategy Recommendation ===")
    print(f"Same surface: {same_surface}")
    print(f"Daily traffic: {daily_traffic:,}")
    print(f"Baseline rate: {baseline_rate:.2%}")
    print(f"MDE (relative): {mde:.0%}")
    print(f"Max duration: {max_days} days")
    print()
    
    if not same_surface:
        print("RECOMMENDATION: Run tests overlapping (independently)")
        print("Reason: Different surfaces → no interaction risk.")
        return
    
    # Check if full factorial is feasible
    n_per_arm = calculate_sample_size_per_variant(baseline_rate, mde, num_comparisons=3)
    total_factorial = n_per_arm * 4
    days_factorial = int(np.ceil(total_factorial / daily_traffic))
    
    # Check if 2-arm sequential is feasible
    n_per_arm_2 = calculate_sample_size_per_variant(baseline_rate, mde, num_comparisons=1)
    total_sequential = n_per_arm_2 * 2 * 2  # Two tests
    days_sequential = int(np.ceil(n_per_arm_2 * 2 / daily_traffic)) * 2
    
    # Mutual exclusion
    days_mutual_excl = int(np.ceil(n_per_arm_2 * 2 / (daily_traffic / 2)))
    
    print(f"Duration estimates:")
    print(f"  Full factorial (4-arm): {days_factorial} days")
    print(f"  Sequential (2 x 2-arm): {days_sequential} days")
    print(f"  Mutual exclusion: {days_mutual_excl} days")
    print()
    
    if days_factorial <= max_days:
        print("RECOMMENDATION: Full Factorial")
        print(f"Reason: Fits within {max_days}-day window ({days_factorial} days).")
        print("Benefit: You can detect interactions and learn the most.")
    elif days_mutual_excl <= max_days:
        print("RECOMMENDATION: Mutual Exclusion")
        print(f"Reason: Full factorial too slow ({days_factorial} days), but mutual exclusion")
        print(f"fits ({days_mutual_excl} days). No interaction risk.")
    else:
        print("RECOMMENDATION: Sequential Testing")
        print(f"Reason: Traffic is too low for concurrent approaches within {max_days} days.")
        print(f"Run the higher-priority test first.")

# Example 1: Listing detail page (high traffic)
print("--- Scenario 1: Listing Detail Page ---")
recommend_concurrent_strategy(
    same_surface=True,
    daily_traffic=15000,
    baseline_rate=0.03,
    mde=0.10,
    max_days=28
)

print("\n")

# Example 2: Host dashboard (low traffic)
print("--- Scenario 2: Host Dashboard ---")
recommend_concurrent_strategy(
    same_surface=True,
    daily_traffic=2000,
    baseline_rate=0.15,
    mde=0.05,
    max_days=28
)

print("\n")

# Example 3: Different surfaces
print("--- Scenario 3: Search Page + Checkout Page ---")
recommend_concurrent_strategy(
    same_surface=False,
    daily_traffic=20000,
    baseline_rate=0.05,
    mde=0.10,
    max_days=28
)

---

## 7. Adapted `calculate_results` for Multi-arm Tests

Below is a more flexible version of `calculate_results` that works with any variant names, not just "control"/"treatment".

In [ ]:
def calculate_results(df, metric, variant_col='variant', control_name='control', alpha=0.05):
    """
    Flexible A/B test results calculator that handles multi-arm experiments.
    
    Compares each non-control variant to the control group.
    Applies Bonferroni correction when there are multiple treatments.
    
    Parameters
    ----------
    df : DataFrame
        Experiment data with variant assignments and metric values
    metric : str
        Column name for the metric to analyze
    variant_col : str
        Column name containing variant assignments
    control_name : str
        Value in variant_col that identifies the control group
    alpha : float
        Overall family-wise significance level
    
    Returns
    -------
    dict with 'summary' DataFrame and 'details' dict
    """
    # Identify treatments: everything that is NOT control
    all_variants = df[variant_col].unique()
    treatments = [v for v in all_variants if v != control_name]
    num_comparisons = len(treatments)
    
    # Bonferroni-adjusted alpha
    adjusted_alpha = alpha / num_comparisons if num_comparisons > 1 else alpha
    z_crit = norm.ppf(1 - adjusted_alpha / 2)
    
    # Control group statistics
    control_data = df[df[variant_col] == control_name][metric]
    n_c = len(control_data)
    mean_c = control_data.mean()
    std_c = control_data.std()
    
    results_list = []
    
    for treatment in sorted(treatments):
        treat_data = df[df[variant_col] == treatment][metric]
        n_t = len(treat_data)
        mean_t = treat_data.mean()
        std_t = treat_data.std()
        
        # For proportions (binary metric)
        if set(df[metric].unique()).issubset({0, 1}):
            se = np.sqrt(mean_c * (1 - mean_c) / n_c + mean_t * (1 - mean_t) / n_t)
        else:
            # For continuous metrics
            se = np.sqrt(std_c**2 / n_c + std_t**2 / n_t)
        
        diff = mean_t - mean_c
        relative_lift = diff / mean_c if mean_c != 0 else np.nan
        z_stat = diff / se if se > 0 else 0
        p_value = 2 * (1 - norm.cdf(abs(z_stat)))
        ci_lower = diff - z_crit * se
        ci_upper = diff + z_crit * se
        significant = p_value < adjusted_alpha
        
        results_list.append({
            'treatment': treatment,
            'n_control': n_c,
            'n_treatment': n_t,
            'mean_control': mean_c,
            'mean_treatment': mean_t,
            'absolute_diff': diff,
            'relative_lift': relative_lift,
            'std_error': se,
            'z_statistic': z_stat,
            'p_value': p_value,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'significant_at_adjusted_alpha': significant
        })
    
    summary_df = pd.DataFrame(results_list)
    
    return {
        'summary': summary_df,
        'num_comparisons': num_comparisons,
        'original_alpha': alpha,
        'adjusted_alpha': adjusted_alpha,
        'control_name': control_name,
        'control_mean': mean_c,
        'control_n': n_c
    }


# Usage example with our FamilyNest factorial experiment
results = calculate_results(df, 'booked', variant_col='variant', control_name='control')

print("=== FamilyNest Multi-arm Results ===")
print(f"Control: {results['control_name']} (n={results['control_n']:,}, rate={results['control_mean']:.4f})")
print(f"Comparisons: {results['num_comparisons']}")
print(f"Adjusted alpha (Bonferroni): {results['adjusted_alpha']:.4f}")
print()

for _, row in results['summary'].iterrows():
    sig = "SIGNIFICANT" if row['significant_at_adjusted_alpha'] else "not significant"
    print(f"  {row['treatment']}:")
    print(f"    Rate: {row['mean_treatment']:.4f} | Lift: {row['relative_lift']:+.2%} | p={row['p_value']:.4f} | {sig}")
    print(f"    95% CI for diff: [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]")
    print()

In [ ]:
# Another common pattern: filtering for specific variant comparisons

# When you only want to compare specific treatments (not all)
# For example: just compare treatment_a vs control (ignore others)

df_subset = df[df['variant'].isin(['control', 'treatment_a'])]
results_single = calculate_results(df_subset, 'booked', control_name='control')

print("=== Single Comparison (no Bonferroni needed) ===")
print(f"Adjusted alpha: {results_single['adjusted_alpha']:.4f} (no correction for 1 comparison)")
print()
row = results_single['summary'].iloc[0]
print(f"Photo carousel vs control:")
print(f"  Lift: {row['relative_lift']:+.2%}")
print(f"  p-value: {row['p_value']:.4f}")
print(f"  Significant: {row['significant_at_adjusted_alpha']}")

In [ ]:
# Full reporting template for stakeholders

def generate_experiment_report(df, metric, experiment_name, variant_descriptions, control_name='control'):
    """
    Generate a formatted experiment report for stakeholders.
    
    Parameters
    ----------
    df : DataFrame
    metric : str
    experiment_name : str
    variant_descriptions : dict
        Maps variant names to human-readable descriptions
    control_name : str
    """
    results = calculate_results(df, metric, control_name=control_name)
    
    print("=" * 60)
    print(f"EXPERIMENT REPORT: {experiment_name}")
    print("=" * 60)
    print()
    print(f"Metric: {metric}")
    print(f"Total users: {len(df):,}")
    print(f"Number of variants: {len(df['variant'].unique())}")
    print(f"Statistical approach: Bonferroni-corrected (alpha={results['adjusted_alpha']:.4f})")
    print()
    print("-" * 60)
    print("RESULTS BY VARIANT")
    print("-" * 60)
    print()
    
    # Control
    desc = variant_descriptions.get(control_name, control_name)
    print(f"  CONTROL: {desc}")
    print(f"    n = {results['control_n']:,} | rate = {results['control_mean']:.4f}")
    print()
    
    # Each treatment
    for _, row in results['summary'].iterrows():
        desc = variant_descriptions.get(row['treatment'], row['treatment'])
        sig = "WINNER" if (row['significant_at_adjusted_alpha'] and row['relative_lift'] > 0) else \
              "LOSER" if (row['significant_at_adjusted_alpha'] and row['relative_lift'] < 0) else \
              "INCONCLUSIVE"
        
        print(f"  {row['treatment'].upper()}: {desc}  [{sig}]")
        print(f"    n = {row['n_treatment']:,} | rate = {row['mean_treatment']:.4f}")
        print(f"    Relative lift: {row['relative_lift']:+.2%}")
        print(f"    Confidence interval: [{row['ci_lower']:+.4f}, {row['ci_upper']:+.4f}]")
        print(f"    p-value: {row['p_value']:.4f}")
        print()
    
    print("-" * 60)
    print("RECOMMENDATION")
    print("-" * 60)
    winners = results['summary'][results['summary']['significant_at_adjusted_alpha'] & 
                                  (results['summary']['relative_lift'] > 0)]
    if len(winners) == 0:
        print("  No variant showed a statistically significant improvement.")
        print("  Consider: extending the test, trying a bolder change, or")
        print("  accepting the null hypothesis and moving on.")
    else:
        best = winners.loc[winners['relative_lift'].idxmax()]
        desc = variant_descriptions.get(best['treatment'], best['treatment'])
        print(f"  Ship: {desc}")
        print(f"  Expected lift: {best['relative_lift']:+.2%}")
    print()


# Generate report for our FamilyNest experiment
variant_descriptions = {
    'control': 'Current listing page (no changes)',
    'treatment_a': 'Kid-focused photo carousel',
    'treatment_b': 'Family amenities section moved higher',
    'treatment_ab': 'Both: photo carousel + amenities section'
}

generate_experiment_report(
    df, 
    'booked', 
    'Listing Page Family Experience (Full Factorial)',
    variant_descriptions
)

---

## Key Takeaways

1. **Not all concurrent tests are risky** — only those touching the same surface need special handling

2. **Full factorial is the gold standard** when you have enough traffic — it lets you detect interactions and learn the most

3. **Interactions matter** — if Change A's effect depends on Change B, you cannot interpret them independently

4. **Bonferroni correction** is simple but conservative — use it when making multiple comparisons to avoid false positives

5. **Always pre-specify** your comparisons and correction method in the experiment plan

6. **The `calculate_results` function** adapts easily to multi-arm tests by filtering on `variant != 'control'` and applying corrections

---

### Quick Reference: When to Use What

```
Traffic > 4x normal needs?  →  Full factorial
Traffic ~ 2x normal needs?  →  Mutual exclusion
Traffic limited?            →  Sequential
Different surfaces?         →  Just overlap them
```